# 04. 취업 지원 Multi-Agent 자율 미션
**프로젝트 목표:** 채용공고와 지원자 이력서를 분석해 직무 적합성, 부족 역량, 면접 준비, 취업 전략을 제안한다. 이 Notebook은 설계·구현 공간이며 완성 코드를 제공하지 않는다.

**필수 조건:** 전문 Agent 3개 이상, Agent별 역할·목표·입출력, Sequential/Parallel/Hybrid 구조 선택, Tool Calling 1개 이상, `StateGraph`, 결과 전달·통합, 테스트 데이터 검증.


## 1. 핵심 개념과 아키텍처 설계 공간
먼저 데이터 의존성을 그린다. 예를 들어 직무 적합성 분석과 면접 질문 생성이 같은 입력을 독립적으로 읽는지, 면접 질문이 적합성 분석 결과를 필요로 하는지 판단한다.

```text
START → [직접 설계] → [직접 설계] → END
```
**작성:** 선택한 구조의 Node 이름과 Edge를 이 칸에 적는다. 병렬 분기가 있으면 합류 시점을 표시한다.


## 2. Import와 LLM 설정
구현할 때 `.env`의 `OPENAI_API_KEY`를 사용하고 기본 모델은 `gpt-4o-mini`로 설정한다. 아래 코드는 설치 확인만 수행한다. 완성 코드가 아니므로 API 호출은 없다.


In [ ]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4o-mini"
print("사용할 모델:", MODEL_NAME)


## 3. 가상 테스트 데이터
아래 채용공고와 이력서는 교육용 가상 사례다. 근거가 없는 학력·경력·자격을 만들어 평가하지 않는다.


In [ ]:
job_posting = {
        "title": "주니어 AI 개발자",
        "company": "가상 데이터서비스 기업",
        "responsibilities": ["Python 데이터 처리", "LLM API를 활용한 기능 개발", "평가 결과 기록"],
        "required_skills": ["Python", "SQL", "Git", "REST API"],
        "preferred_skills": ["LangGraph", "Docker", "클라우드 배포"],
    }

candidate_resume = {
        "name": "가상 지원자 A",
        "projects": ["Python으로 문서 분류 모델 구현", "LangGraph 기반 챗봇 미니 프로젝트"],
        "skills": ["Python", "Git", "LangGraph"],
        "experience": "교육 과정 팀 프로젝트 2회",
    }

test_input = {"job_posting": job_posting, "candidate_resume": candidate_resume}
print(job_posting["title"], "—", candidate_resume["name"])


## 4. Agent 역할 설계 템플릿
먼저 표를 채운 뒤 코드를 작성한다.

| Agent 이름 | 목표 | 입력 Key | 출력 Key | 도구·사용 이유 |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO | TODO |

최소 한 Agent는 명시적인 도구 호출 요청과 실제 실행을 구분해 구현한다. 도구가 필요 없는 Agent에는 억지로 붙이지 않는다.


## 5. State 설계 템플릿
각 Key의 작성 책임자를 표에 대응시킨다. 병렬 Node가 같은 Key에 동시에 쓰는지 확인한다.


In [ ]:
class JobSupportState(TypedDict, total=False):
        job_posting: dict
        candidate_resume: dict
        # TODO: Agent 간 전달할 결과 Key를 추가한다.
        pass


## 6. Tool 및 Agent 구현 공간
예시 도구 후보: 요구역량과 보유역량의 차집합 계산. 도구 호출을 하는 Agent는 모델의 호출 요청을 받아 `ToolNode` 또는 명시적인 도구 실행 단계로 연결한다. 아래 셀은 의도적으로 빈 템플릿이다.


In [ ]:
# TODO: @tool을 사용해 계산 도구를 작성한다.
# TODO: 최소 3개 Agent 함수를 작성한다.
# TODO: 각 함수는 자신이 맡은 State Key만 반환한다.
...


## 7. Graph 구성·Compile·Invoke 공간
설계한 구조에 맞춰 Node와 Edge를 채운다. 병렬 구조라면 모든 필수 결과가 모인 다음 종합 Agent가 실행되는지 확인한다.


In [ ]:
builder = StateGraph(JobSupportState)

# TODO: builder.add_node(...)
# TODO: builder.add_edge(START, ...)
# TODO: 순차·병렬·합류 Edge를 설계한다.
# TODO: graph = builder.compile()
# TODO: result = graph.invoke(test_input)
...


## 8. 실행 결과 검증 시나리오
1. 기본 데이터: Python·Git·LangGraph는 보유, SQL·REST API는 부족하다. `preferred_skills`는 필수 역량과 구분한다.
2. 모든 필수 역량을 가진 지원자: 부족 역량 목록은 비어야 한다.
3. 이력서에 없는 경력을 요구하는 공고: 추정 경력으로 채우지 않는다.
4. 병렬 구조: 두 분기 결과가 모두 준비된 뒤 전략을 작성한다.

AI 코딩 도구를 사용할 때는 먼저 Agent 설계표와 State 계약을 제시하고, 각 Node의 입력·출력 및 검증 코드를 검토한다.


In [ ]:
all_skills_resume = {**candidate_resume, "skills": job_posting["required_skills"]}
test_cases = [
    ("기본", test_input),
    ("필수 역량 충족", {"job_posting": job_posting, "candidate_resume": all_skills_resume}),
]

# TODO: Graph 구현 후 아래 검증을 활성화한다.
# for name, case in test_cases:
#     result = graph.invoke(case)
#     print(name, result)
print("준비된 테스트:", [name for name, _ in test_cases])


## 9. 평가 체크리스트와 제출물
- [ ] 전문 Agent 3개 이상과 각 책임이 설계표에 명확하다.
- [ ] 최소 1개 Agent가 실제 Tool Calling 흐름을 수행한다.
- [ ] State Key와 Node의 읽기·쓰기 관계가 일치한다.
- [ ] Graph가 Compile·Invoke되고, 결과가 종합 Agent까지 전달된다.
- [ ] 기본·필수 역량 충족 사례의 결과를 설명할 수 있다.
- [ ] 근거 없는 경력·역량을 만들어내지 않는다.

**제출:** Agent 설계표, 아키텍처, 실행 가능한 Notebook, 취업 컨설팅 결과, 실행 검증 및 개선 사항.
